In [22]:
#Load the dataset
import pandas as pd

df = pd.read_csv("../data/Womens Clothing E-Commerce Reviews.csv")

print(df.shape)
df.head()

(23486, 11)


,Unnamed: 0,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,0,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
1,1,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
2,2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
3,3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
4,4,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses


In [23]:
#Check missing values

missing = df.isna().sum()
print(missing[missing > 0])

Title              3810
Review Text         845
Division Name        14
Department Name      14
Class Name           14
dtype: int64


In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23486 entries, 0 to 23485
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   Unnamed: 0               23486 non-null  int64 
 1   Clothing ID              23486 non-null  int64 
 2   Age                      23486 non-null  int64 
 3   Title                    19676 non-null  object
 4   Review Text              22641 non-null  object
 5   Rating                   23486 non-null  int64 
 6   Recommended IND          23486 non-null  int64 
 7   Positive Feedback Count  23486 non-null  int64 
 8   Division Name            23472 non-null  object
 9   Department Name          23472 non-null  object
 10  Class Name               23472 non-null  object
dtypes: int64(6), object(5)
memory usage: 2.0+ MB


In [25]:
df.columns

Index(['Unnamed: 0', 'Clothing ID', 'Age', 'Title', 'Review Text', 'Rating',
       'Recommended IND', 'Positive Feedback Count', 'Division Name',
       'Department Name', 'Class Name'],
      dtype='object')

In [26]:
df.describe(include="all")

,Unnamed: 0,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
count,23486.000000,23486.000000,23486.000000,19676,22641,23486.000000,23486.000000,23486.000000,23472,23472,23472
unique,NaN,NaN,NaN,13993,22634,NaN,NaN,NaN,3,6,20
top,NaN,NaN,NaN,Love it!,Perfect fit and i've gotten so many compliment...,NaN,NaN,NaN,General,Tops,Dresses
freq,NaN,NaN,NaN,136,3,NaN,NaN,NaN,13850,10468,6319
mean,11742.500000,918.118709,43.198544,NaN,NaN,4.196032,0.822362,2.535936,NaN,NaN,NaN
std,6779.968547,203.298980,12.279544,NaN,NaN,1.110031,0.382216,5.702202,NaN,NaN,NaN
min,0.000000,0.000000,18.000000,NaN,NaN,1.000000,0.000000,0.000000,NaN,NaN,NaN
25%,5871.250000,861.000000,34.000000,NaN,NaN,4.000000,1.000000,0.000000,NaN,NaN,NaN
50%,11742.500000,936.000000,41.000000,NaN,NaN,5.000000,1.000000,1.000000,NaN,NaN,NaN
75%,17613.750000,1078.000000,52.000000,NaN,NaN,5.000000,1.000000,3.000000,NaN,NaN,NaN


In [27]:
#Apply the required cleaning

df["Review Text"] = df["Review Text"].fillna("")

for col in ["Division Name", "Department Name", "Class Name"]:
    df[col] = df[col].fillna("Unknown")

In [28]:
#Create the product table

products = (
    df.groupby("Clothing ID", as_index=False)
    .agg({
        "Division Name": lambda x: " / ".join(sorted(x.dropna().unique())),
        "Department Name": "first",
        "Class Name": "first"
    })
)

print("Product table rows:", len(products))
print("Unique Clothing IDs:", products["Clothing ID"].nunique())

Product table rows: 1206
Unique Clothing IDs: 1206


In [29]:
#Create the review table

reviews = df[
    [
        "Unnamed: 0",
        "Clothing ID",
        "Age",
        "Title",
        "Review Text",
        "Rating",
        "Recommended IND",
        "Positive Feedback Count"
    ]
].copy()

reviews = reviews.rename(columns={"Unnamed: 0": "Review ID"})

print("Review table rows:", len(reviews))

Review table rows: 23486


In [30]:
#Clothing ID consistency check

conflicts = (
    df.groupby("Clothing ID")[["Department Name", "Class Name"]]
    .nunique()
)

conflicts = conflicts[
    (conflicts["Department Name"] > 1) |
    (conflicts["Class Name"] > 1)
]

print("Number of conflicting Clothing IDs:", len(conflicts))
print(conflicts)

Number of conflicting Clothing IDs: 1
             Department Name  Class Name
Clothing ID                             
1119                       1           2


In [31]:
display(
    df[df["Clothing ID"] == 1119][
        ["Clothing ID", "Division Name", "Department Name", "Class Name"]
    ].drop_duplicates()
)

,Clothing ID,Division Name,Department Name,Class Name
5761,1119,General Petite,Jackets,Jackets
11533,1119,General,Jackets,Outerwear


In [32]:
#Rating check

invalid_ratings = ~df["Rating"].isin([1, 2, 3, 4, 5])

print("Invalid ratings:", invalid_ratings.sum())

Invalid ratings: 0


In [33]:
#Recommended IND check

invalid_recommended = ~df["Recommended IND"].isin([0, 1])

print("Invalid Recommended IND:", invalid_recommended.sum())

Invalid Recommended IND: 0


In [34]:
#Save the two required Part A outputs

products.to_csv("../outputs/products.csv", index=False)
reviews.to_csv("../outputs/cleaned_reviews.csv", index=False)

print("Saved products.csv")
print("Saved cleaned_reviews.csv")

Saved products.csv
Saved cleaned_reviews.csv
